# Claimify
Extract factual assertions from Markdown with one H1 title and H2 sections. Review claims, sentence outcomes, and model stages as DataFrames.

In [4]:
from utils import sentence_rows, setup_notebook, to_dataframe, write_json

from IPython.display import display

from data_platform.digest.claimify import extract_claims, parse_document

project_root = setup_notebook()


## 1. Configure and inspect the source

In [5]:
input_path = project_root / ".data/segmented/article-en-01.md"
run_live = True
question = None
decision_model = "typesafe:jev-latest"
generation_model = "openai:gpt-6-luna"
max_concurrency = 20
markdown_doc = input_path.read_text(encoding="utf-8")
document = parse_document(markdown_doc)
sentences_df = to_dataframe(
    sentence_rows(document),
    columns=["index", "section", "text", "start", "end"],
)
display(sentences_df)


,index,section,text,start,end
0,0,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"Hanoi, Vietnam, Vingroup - Vietnam’s largest private conglomerate, and Legends Global, the premier partner to the wo...",214,481
1,1,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"The agreement marks a significant milestone ahead of the landmark venue’s scheduled opening on July 31, 2027.",481,591
2,2,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"Under the agreement, Vingroup and Legends Global will work together to establish VinFast – Trong Dong Stadium as a p...",593,796
3,3,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"In addition to helping attract major international events to the stadium, Legends Global will collaborate with Ving...",796,1115
4,4,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"Drawing on its extensive experience managing more than 450 venues worldwide, hosting over 20,000 major events, and w...",1117,1423
5,5,"Mr\. Nguyen Viet Quang\, Vice Chairman and CEO of Vingroup\, described VinFast – Trong Dong Stadium as more than a s...","Commenting on the strategic partnership, Mr. Nguyen Viet Quang, Vice Chairman and CEO of Vingroup, said: “VinFast - ...",1561,1730
6,6,"Mr\. Nguyen Viet Quang\, Vice Chairman and CEO of Vingroup\, described VinFast – Trong Dong Stadium as more than a s...","With its world-leading scale, contemporary architecture inspired by Vietnamese cultural heritage, automated retract...",1730,2071
7,7,"Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietn...","Mr. Dan Levy, CEO of Legends Global, commented: “VinFast – Trong Dong Stadium is an extraordinary project for Vietna...",2198,2403
8,8,"Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietn...",Legends Global is honored to partner with Vingroup on this landmark development.,2403,2484
9,9,"Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietn...",We look forward to sharing our expertise in venue and event management to help bring Vingroup’s ambition and vision...,2484,2642


## 2. Extract and review
The source column links each claim to its original sentence. Empty tables mean extraction is disabled or returned no results.

In [ ]:
result = None
if run_live:
    result = await extract_claims(
        markdown_doc,
        question=question,
        decision_model=decision_model,
        generation_model=generation_model,
        max_concurrency=max_concurrency,
    )

claims_df = to_dataframe(
    result.claims if result is not None else [],
    source=markdown_doc,
    columns=["text", "source", "section_index", "sentence_index", "start", "end"],
)
outcomes_df = to_dataframe(
    result.sentences if result is not None else [],
    columns=["index", "status", "text", "selected_text", "clarified_text"],
)
display(claims_df)
display(outcomes_df)


## 3. Inspect stages
Choose a sentence index to review decisions and repair attempts. Provider confidence is metadata, not a correctness guarantee.

In [ ]:
stages_df = to_dataframe(
    [
        {"sentence_index": sentence.index, **stage.model_dump(mode="json")}
        for sentence in (result.sentences if result is not None else [])
        for stage in sentence.stages
    ],
    columns=["sentence_index", "stage", "output", "model", "confidence", "usage"],
)
sentence_index = 0
display(stages_df.loc[stages_df["sentence_index"] == sentence_index])
display(outcomes_df.groupby("status").size().rename("sentences").to_frame())


## 4. Export (optional)
Saving replaces the claim CSV and full JSON export. JSON retains source text, stage traces, and usage.

In [ ]:
save_output = False
if save_output and result is not None:
    output_dir = project_root / ".data/claimify"
    output_dir.mkdir(parents=True, exist_ok=True)
    claims_df.to_csv(output_dir / f"{input_path.stem}.claims.csv", index=False)
    payload = {
        "source_markdown": markdown_doc,
        "question": question,
        "result": result.model_dump(mode="json"),
    }
    write_json(output_dir / f"{input_path.stem}.json", payload)
